# Automatización Fase 4 (Consolidación) — Matriz con detalle de vuelos + CUADRO FINAL en Archivo 10

Continúa después de `Automatizacion_Fase3_Asignacion_Instructor_Vuelos.ipynb` (que ya escribió
instructor + "LCK A320F" en los bloques). Este notebook reconstruye los mismos bloques/asignaciones
(mismo BigQuery + mismo matching contra la Matriz, sin cambios) y agrega dos escrituras nuevas,
**ambas en modo vista previa primero**:

1. **Reemplazar el slot `"LCK A320F"` de la Matriz** por el detalle de vuelos, formato confirmado
   por Fernando:
   ```
   LCK A320F
   LIM-CUZ-LIM
   LA 2013 (08:40-10:00 hrs)
   LA 2014 (10:40-12:15 hrs)
   LCK A320F
   LIM-CUZ-LIM
   LA 2028 (13:25-14:45 hrs)
   LA 2029 (15:30-17:05 hrs)
   ```
   (las 2 mitades del bloque, una debajo de otra, en la MISMA celda).
2. **Armar el CUADRO FINAL en Archivo 10** (hoja "LCK 320", bloque rotulado en AC8, encabezados en
   fila 9, datos desde fila 10): Fecha, DíaSEM, Vuelo, Ruta, N° Cupos, Grupo, INS — filtrando de
   las asignaciones solo `"LCK A320F"` y `"LCK A320F + Habilitación A320F"` (pasos 17-19 del
   documento).

**Lo que sigue SIN automatizar (confirmado, no se inventa):** la columna "Grupo" **por
tripulante** (pasos 19-20, columnas INS FINAL / INS F a considerar / Grupo del Archivo 10) sigue
siendo manual — necesita el historial de REVA por tripulante, que no está confirmado en BigQuery.
Tampoco se automatiza el paso 21 (validación de capacidad por grupo) más allá de un conteo simple
de vuelos por grupo para que lo revises.


## 1. Instalar dependencias y autenticar

In [ ]:
!pip install -q gspread


In [ ]:
import sys
import re
import unicodedata
import collections
from collections import defaultdict
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
# Mismo motivo que en la Fase 3: la cuenta de Fernando no tiene "bigquery.jobs.create"
# en operations-data-prod, pero sí en datadem-home (proyecto de facturación).
client = bigquery.Client(project="datadem-home")
print("Autenticado: BigQuery + Google Sheets con tu cuenta.")


## 2. Query a BigQuery + armado de candidatos/bloques (igual que Fase 3, sin cambios)

In [ ]:
MES_OBJETIVO = 10
ANIO_OBJETIVO = 2026

WD_ES = {
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
    "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado",
    "Sunday": "domingo",
}

RUTAS_VALIDAS_ARR = {"AQP", "CIX", "CJA", "CUZ", "PEM", "PIU", "TBP", "TPP", "TCQ"}
EXCLUSIONES_MES = {"AQP"}  # confirmado con Fernando: sigue vigente en octubre-2026

HORA_MIN_SALIDA = pd.Timedelta(hours=8, minutes=30)
HBT_MIN = pd.Timedelta(hours=1)
CONEXION_MIN = pd.Timedelta(minutes=50)
CONEXION_MAX = pd.Timedelta(hours=1, minutes=30)
PSV_MAX = pd.Timedelta(hours=11)

QUERY_NB = """
SELECT
  pairing_id                       AS trip,
  flight_start_date_local_time     AS inicio_vuelo_lt,
  duty_calendar_day_number         AS dia_duty,
  flight_number                    AS vuelo,
  departure_airport_code           AS dep,
  arrival_airport_code             AS arr,
  flight_departure_time_crew_base  AS std_hb,
  flight_arrival_hour_block_time   AS sta_hb,
  flight_block_time                AS hbt,
  subfleet_code                    AS sub_fleet

FROM `operations-data-prod.carmen_gold.crew_pairing_carmen_system`

WHERE
  flight_start_date_local_time BETWEEN DATE '2026-10-01' AND DATE '2026-10-31'
  AND subsidiary_code IN ('LP')
  AND load_type_code = 'FP'
  AND crew_range_type_code = 'SAB'
  AND subfleet_code IN ('319', '320')

QUALIFY
  CASE
    WHEN load_type_code = 'FP' AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'FP' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    WHEN load_type_code = 'ES' AND
         MAX(CASE WHEN load_type_code = 'FP' THEN 0 ELSE 0 END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year) = -1 AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'ES' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    ELSE -1
  END = 0

ORDER BY pairing_id ASC
"""

df_raw = client.query(QUERY_NB).to_dataframe(create_bqstorage_client=False)
print(f"Filas descargadas: {len(df_raw)}")


In [ ]:
def parse_hora(s):
    if pd.isna(s):
        return pd.NaT
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def cargar_bq_a_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    df["trip"] = df["trip"].astype(str)
    df["dia_duty"] = df["dia_duty"].astype(int)
    df["fecha_dt"] = pd.to_datetime(df["inicio_vuelo_lt"])
    df["std_td"] = df["std_hb"].apply(parse_hora)
    df["sta_td"] = df["sta_hb"].apply(parse_hora)
    df["hbt_td"] = df["hbt"].apply(parse_hora)
    df["std_dt"] = df["fecha_dt"] + df["std_td"]
    df["sta_dt"] = df["fecha_dt"] + df["sta_td"]
    df.loc[df["sta_dt"] < df["std_dt"], "sta_dt"] += pd.Timedelta(days=1)
    df["dia_semana"] = df["fecha_dt"].dt.day_name().map(WD_ES)
    return df


def separar_instancias_trip(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["trip", "fecha_dt", "std_dt"]).reset_index(drop=True)
    instancia = []
    trip_actual = None
    max_dia_duty = -1
    idx_instancia = 0
    for _, row in df.iterrows():
        if row["trip"] != trip_actual:
            trip_actual = row["trip"]
            idx_instancia = 0
            max_dia_duty = row["dia_duty"]
        elif row["dia_duty"] < max_dia_duty:
            idx_instancia += 1
            max_dia_duty = row["dia_duty"]
        else:
            max_dia_duty = max(max_dia_duty, row["dia_duty"])
        trip_val = row["trip"]
        instancia.append(f"{trip_val}_{idx_instancia}")

    df["trip_original"] = df["trip"]
    df["trip"] = instancia
    return df


def filtrar_mes_y_ruta(df: pd.DataFrame, mes: int, anio: int) -> pd.DataFrame:
    rutas_validas = RUTAS_VALIDAS_ARR - {c.upper() for c in EXCLUSIONES_MES}
    en_mes = (df["fecha_dt"].dt.month == mes) & (df["fecha_dt"].dt.year == anio)
    sale_de_lim = df["dep"] == "LIM"
    llega_a_lim = df["arr"] == "LIM"
    ruta_nacional_ok = df["arr"].isin(rutas_validas) | (llega_a_lim)
    return df[en_mes & (sale_de_lim | llega_a_lim) & ruta_nacional_ok].copy()


def armar_primeras_mitades(df: pd.DataFrame, dia_duty_min_real=None):
    validos_rows = []
    excluidos_rows = []

    for trip, grupo in df.groupby("trip"):
        dia_min = grupo["dia_duty"].min()
        if dia_duty_min_real is not None and trip in dia_duty_min_real.index and dia_min != dia_duty_min_real.loc[trip]:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 real fuera de mes/ruta"})
            continue
        dia1 = grupo[grupo["dia_duty"] == dia_min].sort_values("std_dt")

        if len(dia1) < 2:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 sin vuelta el mismo día"})
            continue
        if len(dia1) in (6, 8, 10):
            excluidos_rows.append({"trip": trip, "motivo": f"día 1 tiene {len(dia1)} tramos"})
            continue

        ida, vuelta = dia1.iloc[0], dia1.iloc[1]
        if ida["dep"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el primer tramo no sale de LIM"})
            continue
        if vuelta["arr"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el segundo tramo no vuelve a LIM"})
            continue
        if vuelta["dep"] != ida["arr"]:
            excluidos_rows.append({"trip": trip, "motivo": "ruta triangular"})
            continue

        motivos = []
        if (ida["std_dt"] - ida["fecha_dt"]) <= HORA_MIN_SALIDA:
            motivos.append("sale antes/igual a 08:30")
        if ida["hbt_td"] <= HBT_MIN:
            motivos.append("HBT ida <= 1h")
        if vuelta["hbt_td"] <= HBT_MIN:
            motivos.append("HBT vuelta <= 1h")
        conexion = vuelta["std_dt"] - ida["sta_dt"]
        if conexion <= pd.Timedelta(0):
            motivos.append("conexión interna negativa/cero")
        psv = vuelta["sta_dt"] - ida["std_dt"]
        if psv > PSV_MAX:
            motivos.append(f"PSV {psv} > 11h")

        if motivos:
            excluidos_rows.append({"trip": trip, "motivo": "; ".join(motivos)})
            continue

        validos_rows.append({
            "Fecha": ida["fecha_dt"].strftime("%d/%m/%Y"),
            "DíaSem": ida["dia_semana"],
            "Pairing ID": trip,
            "Vuelo Ida": ida["vuelo"], "Dep": ida["dep"], "Arr": ida["arr"],
            "STD Ida": ida["std_hb"], "STA Ida": ida["sta_hb"], "HBT Ida": ida["hbt"],
            "Vuelo Vuelta": vuelta["vuelo"], "Dep Vta": vuelta["dep"], "Arr Vta": vuelta["arr"],
            "STD Vuelta": vuelta["std_hb"], "STA Vuelta": vuelta["sta_hb"], "HBT Vuelta": vuelta["hbt"],
            "Conexión": str(conexion), "PSV Total": str(psv),
            "Sub Flota": ida["sub_fleet"],
            "_orden": ida["std_dt"],
        })

    cols_finales = ["Fecha", "DíaSem", "Pairing ID", "Vuelo Ida", "Dep", "Arr", "STD Ida",
                     "STA Ida", "HBT Ida", "Vuelo Vuelta", "Dep Vta", "Arr Vta", "STD Vuelta",
                     "STA Vuelta", "HBT Vuelta", "Conexión", "PSV Total", "Sub Flota"]

    if not validos_rows:
        return pd.DataFrame(columns=cols_finales), pd.DataFrame(excluidos_rows)

    validos = pd.DataFrame(validos_rows).sort_values("_orden")
    validos = validos[cols_finales].reset_index(drop=True)
    return validos, pd.DataFrame(excluidos_rows)


def _fecha_dt(s):
    d, m, a = s.split("/")
    return pd.Timestamp(year=int(a), month=int(m), day=int(d))


def _hora_td(s):
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def parear_candidatos(validos: pd.DataFrame):
    df = validos.copy()
    df["_ida_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STD Ida"]), axis=1)
    df["_vta_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STA Vuelta"]), axis=1)
    df = df.sort_values("_ida_dt").reset_index(drop=True)

    usados = set()
    bloques = []
    for i, row in df.iterrows():
        if row["Pairing ID"] in usados:
            continue
        ventana_ini = row["_vta_dt"] + CONEXION_MIN
        ventana_fin = row["_vta_dt"] + CONEXION_MAX
        mismo_dia = df[
            (~df["Pairing ID"].isin(usados)) &
            (df["Pairing ID"] != row["Pairing ID"]) &
            (df["Fecha"] == row["Fecha"]) &
            (df["_ida_dt"] > ventana_ini) & (df["_ida_dt"] < ventana_fin) &
            (df["_vta_dt"] - row["_ida_dt"] <= PSV_MAX)
        ].sort_values("_ida_dt")

        usados.add(row["Pairing ID"])
        if len(mismo_dia) > 0:
            segunda = mismo_dia.iloc[0]
            usados.add(segunda["Pairing ID"])
            bloques.append((row, segunda))
        else:
            bloques.append((row, None))
    return bloques


df = cargar_bq_a_df(df_raw)
df = separar_instancias_trip(df)
dia_duty_min_real = df.groupby("trip")["dia_duty"].min()
df_filtrado = filtrar_mes_y_ruta(df, MES_OBJETIVO, ANIO_OBJETIVO)
validos, excluidos = armar_primeras_mitades(df_filtrado, dia_duty_min_real)
bloques = parear_candidatos(validos)

n_parejas = sum(1 for _, b in bloques if b is not None)
n_solos = sum(1 for _, b in bloques if b is None)
print(f"Candidatos válidos: {len(validos)}")
print(f"Bloques armados: {len(bloques)} ({n_parejas} completos, {n_solos} solos)")


## 3. Leer la Matriz real (con la fila/columna EXACTA de cada slot "LCK A320F")

Igual que en la Fase 3, pero ahora también se guarda `fila_hoja`/`col_hoja` de cada slot -> se
necesita para poder escribir de vuelta en la celda correcta.

In [ ]:
URL_MATRIZ = "https://docs.google.com/spreadsheets/d/19WmwaoLDZnArNztu_dJNwi7bjrGq-_cx_gw0aN96zfk/edit?gid=580414308"

sh_matriz = gc.open_by_url(URL_MATRIZ)
ws_matriz = sh_matriz.get_worksheet_by_id(580414308)

FILA_ENCABEZADO_FECHAS = 2
FILA_PRIMER_INSTRUCTOR = 3
COL_PRIMERA_FECHA = 3

valores_m = ws_matriz.get_all_values()

fila_fechas = valores_m[FILA_ENCABEZADO_FECHAS - 1]
fechas_matriz = []
for celda in fila_fechas[COL_PRIMERA_FECHA - 1:]:
    fechas_matriz.append(celda.strip() if celda.strip() else None)

reservas = []  # (bp, nombre_matriz, fecha_str, fila_hoja, col_hoja)
for i, fila in enumerate(valores_m[FILA_PRIMER_INSTRUCTOR - 1:]):
    fila_hoja = FILA_PRIMER_INSTRUCTOR + i
    if not fila or not fila[0].strip():
        continue
    bp = fila[0].strip().lstrip("'")
    nombre = fila[1].strip() if len(fila) > 1 else ""
    for j, celda in enumerate(fila[COL_PRIMERA_FECHA - 1:]):
        col_hoja = COL_PRIMERA_FECHA + j
        if celda.strip().upper() == "LCK A320F":
            fecha_str = fechas_matriz[j] if j < len(fechas_matriz) else None
            if fecha_str:
                reservas.append((bp, nombre, fecha_str, fila_hoja, col_hoja))

print(f"Slots 'LCK A320F' encontrados en la Matriz: {len(reservas)}")


## 4. Emparejar reservas con bloques (misma lógica de la Fase 3)

Mismas reglas confirmadas: solo bloque completo en la fecha exacta; nombre normalizado
(tildes/mayúsculas); si algo no calza, se reporta, no se inventa.

In [ ]:
INSTRUCTORES_DATA = [
    ("Christian Rondon", "1271571", " RONDON BARRUTIA CHRISTIAN ERIC "),
    ("Erika Davila", "967092", "DAVILA BELLO MARIA ERIKA"),
    ("Sebastian Correa", "2396710", " CORREA GARCIA JUAN SEBASTIAN "),
    ("Fiorella Ruiz", "2713993", "RUIZ RIOJA FIORELLA DEL PILAR"),
    ("Jazmin Guerra", "29530", "GUERRA SUAREZ JAZMIN"),
    ("Jennifert Acurio", "3779550", "ACURIO DARGENT JENNIFERT MILAGROS"),
    ("Karen Santa Cruz", "2843319", "SANTA CRUZ HUAMAN KAREN"),
    ("Luis Bacigalupo", "2963161", "BACIGALUPO FLORES LUIS ENRIQUE"),
    ("Claudia Flores", "3217561", " FLORES FUENTES DAVILA CLAUDIA ALEXANDRA "),
    ("Karla Moz", "71348", "MOZ MONTES KARLA LISSETTE"),
    ("Elizabeth Torres", "2369641", "TORRES POLO ELIZABETH DEL PILAR"),
    ("Patricia Najar", "2369624", "NAJAR CRUZ PATRICIA DEL PILAR"),
    ("Javier Zapata", "3134911", "ZAPATA GARAYAR JAVIER RICARDO SALVADOR"),
    ("Jefferson Mendez", "3750335", " MENDEZ RUCOBA JEFFERSON "),
    ("Gabriela Ungaro", "3852423", "UNGARO GUTIERREZ GABRIELA"),
    ("Mariella Carrasco", "2604360", "CARRASCO BENAVIDES ROSA MARIELLA"),
    ("Cesar Campos", "2823133", " CAMPOS CONCHE CESAR AUGUSTO "),
    ("Kevin Segovia", "3189967", "SEGOVIA TAPIA RAY KEVIN"),
    ("Milagros Salas", "2415373", "SALAS COSIO MILAGROS PATRICIA"),
    ("Judith Fernandez", "2440915", "FERNANDEZ GARCIA JUDITH JULIET"),
    ("Rafael Nieto", "3796947", " NIETO SAENZ RAFAEL ANTONIO "),
    ("Gianfranco Celiz", "3841387", " CELIZ ROSSI GIANFRANCO PAOLO "),
]


def normalizar_nombre(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.strip().lower()


def normalizar_fecha(fecha_str):
    d, m, a = fecha_str.strip().split("/")
    return (int(d), int(m), int(a))


def emparejar_matriz_con_bloques(reservas, bloques):
    mapa_nombre_canonico = {normalizar_nombre(n): n for n, _bp, _legal in INSTRUCTORES_DATA}

    bloques_por_fecha = defaultdict(list)
    for idx, (candA, _candB) in enumerate(bloques):
        bloques_por_fecha[normalizar_fecha(candA["Fecha"])].append(idx)

    usados_bloques = set()
    asignaciones_por_bloque = {}
    celda_matriz_por_bloque = {}
    sin_bloque = []
    sin_match_nombre = []

    for bp, nombre_matriz, fecha_str, fila_hoja, col_hoja in reservas:
        nombre_canonico = mapa_nombre_canonico.get(normalizar_nombre(nombre_matriz))
        if nombre_canonico is None:
            sin_match_nombre.append((bp, nombre_matriz, fecha_str))
            continue

        fecha_norm = normalizar_fecha(fecha_str)
        candidatos_idx = [i for i in bloques_por_fecha.get(fecha_norm, []) if i not in usados_bloques]
        candidatos_completos = [i for i in candidatos_idx if bloques[i][1] is not None]

        elegido = candidatos_completos[0] if candidatos_completos else None
        if elegido is None:
            sin_bloque.append((bp, nombre_canonico, fecha_str))
            continue

        usados_bloques.add(elegido)
        asignaciones_por_bloque[elegido] = (nombre_canonico, "LCK A320F")
        celda_matriz_por_bloque[elegido] = (fila_hoja, col_hoja)

    return asignaciones_por_bloque, celda_matriz_por_bloque, sin_bloque, sin_match_nombre


asignaciones_por_bloque, celda_matriz_por_bloque, sin_bloque, sin_match_nombre = emparejar_matriz_con_bloques(reservas, bloques)

print(f"Reservas de la Matriz: {len(reservas)}")
print(f"Asignadas a un bloque: {len(asignaciones_por_bloque)}")
print(f"Sin bloque completo esa fecha (revisar a mano): {len(sin_bloque)}")
for bp, nombre, fecha in sin_bloque:
    print(f"  - {nombre} (BP {bp}) reservado el {fecha}")
print(f"Sin match de nombre (revisar a mano): {len(sin_match_nombre)}")
for bp, nombre, fecha in sin_match_nombre:
    print(f"  - '{nombre}' (BP {bp}, {fecha})")


## 5. Vista previa: detalle de vuelos para reemplazar cada slot "LCK A320F" en la Matriz

Formato confirmado por Fernando: dentro de la MISMA celda, las 2 mitades del bloque una
debajo de otra (actividad / ruta / vuelo ida / vuelo vuelta), separadas por salto de línea.

In [ ]:
def construir_texto_matriz(candA, candB, actividad="LCK A320F"):
    partes = []
    for cand in [candA] + ([candB] if candB is not None else []):
        ruta = f"LIM-{cand['Arr']}-LIM"
        leg_ida = f"LA {cand['Vuelo Ida']} ({str(cand['STD Ida'])[:5]}-{str(cand['STA Ida'])[:5]} hrs)"
        leg_vta = f"LA {cand['Vuelo Vuelta']} ({str(cand['STD Vuelta'])[:5]}-{str(cand['STA Vuelta'])[:5]} hrs)"
        partes.extend([actividad, ruta, leg_ida, leg_vta])
    return "\n".join(partes)


actualizaciones_matriz = []  # (fila_hoja, col_hoja, texto)
for idx_bloque, (nombre, actividad) in asignaciones_por_bloque.items():
    candA, candB = bloques[idx_bloque]
    fila_hoja, col_hoja = celda_matriz_por_bloque[idx_bloque]
    texto = construir_texto_matriz(candA, candB, actividad)
    actualizaciones_matriz.append((fila_hoja, col_hoja, texto))

print(f"Celdas de la Matriz a actualizar: {len(actualizaciones_matriz)}\n")
for fila_hoja, col_hoja, texto in actualizaciones_matriz:
    print(f"--- Matriz fila {fila_hoja}, columna {col_hoja} ---")
    print(texto)
    print()


## 6. Escribir en la Matriz real — DESACTIVADO por defecto

In [ ]:
# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA DE LA CELDA ANTERIOR ---

# celdas_gs = [gspread.Cell(row=f, col=c, value=t) for f, c, t in actualizaciones_matriz]
# ws_matriz.update_cells(celdas_gs, value_input_option="USER_ENTERED")
# print(f"Actualizadas {len(celdas_gs)} celdas en la Matriz.")


## 7. Vista previa: CUADRO FINAL para Archivo 10 (pasos 17-19)

Solo se incluyen asignaciones con actividad `"LCK A320F"` o `"LCK A320F + Habilitación A320F"`.
**Cambio confirmado por Fernando:** la columna "Grupo" del CUADRO FINAL (AH) queda **vacía** —
los grupos ya NO los fija este notebook. Fernando los va a escribir a mano en 4 celdas (listas de
nombres separadas por coma): `AD2`=Grupo 1, `AF2`=Grupo 2, `AG2`=Grupo 3, `AH2`=Grupo 4 (opcional).
Más abajo (sección 7b) se arma una tabla auxiliar con una fórmula que lee esas 4 celdas.


In [ ]:
ACTIVIDADES_CUADRO_FINAL = {"LCK A320F", "LCK A320F + Habilitación A320F"}

filas_cuadro_final = []  # dicts: Fecha, DiaSem, Vuelo, Ruta, Cupos, Grupo (vacío), INS

for idx_bloque, (nombre, actividad) in asignaciones_por_bloque.items():
    if actividad not in ACTIVIDADES_CUADRO_FINAL:
        continue
    candA, candB = bloques[idx_bloque]

    for cand in [candA] + ([candB] if candB is not None else []):
        cupos = 4 if str(cand["Sub Flota"]) == "320" else 3
        filas_cuadro_final.append({
            "Fecha": cand["Fecha"],
            "DiaSem": cand["DíaSem"],
            "Vuelo": f"{cand['Vuelo Ida']}/{cand['Vuelo Vuelta']}",
            "Ruta": f"LIM-{cand['Arr']}-LIM",
            "Cupos": cupos,
            "Grupo": "",  # confirmado: se deja vacío, Fernando arma los grupos a mano
            "INS": nombre,
        })

print(f"Filas para el CUADRO FINAL: {len(filas_cuadro_final)}")
print("\nPrimeras filas:")
for f in filas_cuadro_final[:6]:
    print(f)


## 8. Escribir el CUADRO FINAL en Archivo 10 — DESACTIVADO por defecto

In [ ]:
URL_ARCHIVO_10_LCK320F = "https://docs.google.com/spreadsheets/d/1NZN565fOJUtoETQvvPzdHpRvyHp4stY2hsrqtjNjSEU/edit?gid=1262242775"
sh_archivo10 = gc.open_by_url(URL_ARCHIVO_10_LCK320F)
ws_archivo10_lck320f = sh_archivo10.get_worksheet_by_id(1262242775)

FILA_INICIO_CUADRO_FINAL = 10  # confirmado: encabezados en fila 9, datos desde fila 10
COL_INICIO_CUADRO_FINAL = "AC"

valores_a_escribir = [
    [f["Fecha"], f["DiaSem"], f["Vuelo"], f["Ruta"], f["Cupos"], f["Grupo"], f["INS"]]
    for f in filas_cuadro_final
]
rango_cuadro_final = (
    f"{COL_INICIO_CUADRO_FINAL}{FILA_INICIO_CUADRO_FINAL}:"
    f"AI{FILA_INICIO_CUADRO_FINAL + len(valores_a_escribir) - 1}"
)
print(f"Se escribirían {len(valores_a_escribir)} filas en Archivo 10, rango {rango_cuadro_final}")

# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---
# if valores_a_escribir:
#     ws_archivo10_lck320f.update(rango_cuadro_final, valores_a_escribir)
#     print(f"Escritas {len(valores_a_escribir)} filas en Archivo 10.")


## 9. Tabla auxiliar (INS / Cantidad / Grupos) + lista de instructores no considerados

Esto es lo que le da sentido a dejar "Grupo" vacío en el CUADRO FINAL: en vez de que este
notebook decida los grupos, Fernando los escribe a mano (listas separadas por coma) en:
`AD2`=Grupo 1, `AF2`=Grupo 2, `AG2`=Grupo 3, `AH2`=Grupo 4 (confirmado por Fernando).

- **Tabla auxiliar** (`AK14:AM...`, mismo lugar que el `AK15:AM27` del manual original): una fila
  por instructor que SÍ tiene vuelos este mes, con la cantidad de vuelos y una **fórmula** en
  "Grupos" que busca su nombre dentro de esas 4 celdas y devuelve "Grupo 1/2/3/4" (o "SIN GRUPO"
  si no aparece en ninguna). La fórmula se recalcula sola cada vez que Fernando edite AD2/AF2/AG2/AH2.
- **INS NO CONSIDERADOS**: del catálogo de 21 instructores que dio Fernando, los que este mes NO
  tienen ningún vuelo en el CUADRO FINAL (no van a dictar LCK este mes) — lista estática, no
  fórmula, porque depende de los vuelos ya calculados arriba, no de lo que Fernando escriba en
  AD2:AH2.

In [ ]:
CATALOGO_INSTRUCTORES = [
    ("2713993", "Fiorella Ruiz"),
    ("2396710", "Sebastian Correa"),
    ("1271571", "Christian Rondon"),
    ("2843319", "Karen Santa Cruz"),
    ("3779550", "Jennifert Acurio"),
    ("29530", "Jazmin Guerra"),
    ("3217561", "Claudia Flores"),
    ("71348", "Karla Moz"),
    ("967092", "Erika Davila"),
    ("2369641", "Elizabeth Torres"),
    ("2369624", "Patricia Najar"),
    ("3134911", "Javier Zapata"),
    ("3750335", "Jefferson Mendez"),
    ("3852423", "Gabriela Ungaro"),
    ("2604360", "Mariella Carrasco"),
    ("2823133", "Cesar Campos"),
    ("3189967", "Kevin Segovia"),
    ("2415373", "Milagros Salas"),
    ("2440915", "Judith Fernandez"),
    ("3796947", "Rafael Nieto"),
    ("3841387", "Gianfranco Celiz"),
]  # lista dada directamente por Fernando (21 instructores), no derivada de ningún archivo

conteo_vuelos_por_ins = collections.Counter(f["INS"] for f in filas_cuadro_final)
instructores_con_vuelos = sorted(conteo_vuelos_por_ins.keys())
instructores_no_considerados = [nombre for _bp, nombre in CATALOGO_INSTRUCTORES
                                 if nombre not in conteo_vuelos_por_ins]

print(f"Instructores con vuelos en el CUADRO FINAL: {len(instructores_con_vuelos)}")
for nombre in instructores_con_vuelos:
    print(f"  {nombre}: {conteo_vuelos_por_ins[nombre]} vuelos")

print(f"\nInstructores del catálogo SIN ningún vuelo este mes (INS NO CONSIDERADOS): {len(instructores_no_considerados)}")
for nombre in instructores_no_considerados:
    print(f"  {nombre}")

# --- ubicaciones en Archivo 10 ---
FILA_HEADER_AUX = 14   # INS | CANTIDAD | GRUPOS
FILA_INICIO_AUX = 15   # confirmado: igual al AK15 del manual original
FILA_TOTAL_AUX = FILA_INICIO_AUX + len(instructores_con_vuelos)
COL_NO_CONSIDERADOS = "AO"


def formula_grupo(fila_ins):
    ref = f"$AK${fila_ins}"
    return (
        f'=SI(ESNUMERO(HALLAR({ref}; $AD$2)); "Grupo 1"; '
        f'SI(ESNUMERO(HALLAR({ref}; $AE$2)); "Grupo 2"; '
        f'SI(ESNUMERO(HALLAR({ref}; $AF$2)); "Grupo 3"; '
        f'SI(ESNUMERO(HALLAR({ref}; $AG$2)); "Grupo 4"; "SIN GRUPO"))))'
        
    )


filas_tabla_aux = []
for i, nombre in enumerate(instructores_con_vuelos):
    fila_ins = FILA_INICIO_AUX + i
    filas_tabla_aux.append([nombre, conteo_vuelos_por_ins[nombre], formula_grupo(fila_ins)])

print(f"\nTabla auxiliar: encabezado fila {FILA_HEADER_AUX} (AK:AM), datos desde fila {FILA_INICIO_AUX}, TOTAL en fila {FILA_TOTAL_AUX}")
for fila in filas_tabla_aux:
    print(fila)


## 10. Escribir tabla auxiliar + INS NO CONSIDERADOS en Archivo 10 — DESACTIVADO por defecto

In [ ]:
# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---

# ws_archivo10_lck320f.update(f"AK{FILA_HEADER_AUX}:AM{FILA_HEADER_AUX}", [["INS", "CANTIDAD", "GRUPOS"]])
# if filas_tabla_aux:
#     ws_archivo10_lck320f.update(
#         f"AK{FILA_INICIO_AUX}:AM{FILA_TOTAL_AUX - 1}",
#         filas_tabla_aux,
#         value_input_option="USER_ENTERED",  # necesario para que la columna GRUPOS quede como formula, no como texto
#     )
# ws_archivo10_lck320f.update(
#     f"AK{FILA_TOTAL_AUX}:AL{FILA_TOTAL_AUX}",
#     [["TOTAL", f"=SUM(AL{FILA_INICIO_AUX}:AL{FILA_TOTAL_AUX - 1})"]],
#     value_input_option="USER_ENTERED",
# )
# print(f"Tabla auxiliar escrita: {len(filas_tabla_aux)} instructores + fila TOTAL.")

# ws_archivo10_lck320f.update(f"{COL_NO_CONSIDERADOS}{FILA_HEADER_AUX}", [["INS NO CONSIDERADOS (solo nombres)"]])
# if instructores_no_considerados:
#     valores_no_considerados = [[n] for n in instructores_no_considerados]
#     ws_archivo10_lck320f.update(
#         f"{COL_NO_CONSIDERADOS}{FILA_HEADER_AUX + 1}:{COL_NO_CONSIDERADOS}{FILA_HEADER_AUX + len(valores_no_considerados)}",
#         valores_no_considerados,
#     )
#     print(f"'INS NO CONSIDERADOS' escrito: {len(valores_no_considerados)} nombres.")


## 11. QA: recalcular en Python que todo quede consistente

In [ ]:
# cada bloque asignado aparece como maximo 1 vez en el CUADRO FINAL (por pairing, 2 filas por bloque completo)
pares_esperados = sum(1 for _, (n, a) in asignaciones_por_bloque.items() if a in ACTIVIDADES_CUADRO_FINAL) * 2
print("Filas esperadas (2 por bloque con actividad LCK):", pares_esperados)
print("Filas realmente construidas:", len(filas_cuadro_final))
assert pares_esperados == len(filas_cuadro_final)

# ninguna fila del CUADRO FINAL con Cupos fuera de {3,4}
cupos_malos = [f for f in filas_cuadro_final if f["Cupos"] not in (3, 4)]
print("Filas con Cupos fuera de {3,4} (deben ser 0):", len(cupos_malos))

# columna Grupo del CUADRO FINAL debe quedar SIEMPRE vacia (confirmado con Fernando)
grupo_no_vacio = [f for f in filas_cuadro_final if f["Grupo"] != ""]
print("Filas con Grupo NO vacío en el CUADRO FINAL (deben ser 0):", len(grupo_no_vacio))

# cada instructor del catalogo esta en EXACTAMENTE una de las dos listas (con vuelos / no considerado)
nombres_catalogo = {nombre for _bp, nombre in CATALOGO_INSTRUCTORES}
cubiertos = set(instructores_con_vuelos) | set(instructores_no_considerados)
print("Catálogo completamente cubierto (con vuelos + no considerados):", nombres_catalogo == cubiertos)
print("Instructores con vuelos que no estan en el catalogo dado (revisar a mano):",
      set(instructores_con_vuelos) - nombres_catalogo)

# la cantidad de celdas de Matriz a actualizar coincide con la cantidad de bloques asignados
print("Celdas de Matriz a actualizar == bloques asignados:",
      len(actualizaciones_matriz) == len(asignaciones_por_bloque))


## 12. Estado y pendientes — sin inventar nada

### Lo que este notebook automatiza
- Reemplaza el slot `"LCK A320F"` de la Matriz por el detalle real de vuelos (formato confirmado
  por Fernando), en modo vista previa.
- Arma el CUADRO FINAL para Archivo 10 (Fecha, DíaSEM, Vuelo, Ruta, Cupos, INS), filtrando por
  actividad LCK, con la columna Grupo **vacía** (confirmado por Fernando).
- Arma la tabla auxiliar (INS / Cantidad / Grupos, en `AK14:AM...`) con una **fórmula** que lee
  las 4 celdas donde Fernando escribe a mano los grupos (`AD2`/`AF2`/`AG2`/`AH2`), en vez de que
  el notebook decida los grupos.
- Arma la lista "INS NO CONSIDERADOS" (instructores del catálogo sin ningún vuelo este mes).

### Lo que sigue sin automatizar (confirmado, necesita datos que no existen/no están confirmados)
- **Columna "Grupo" por TRIPULANTE** (no por vuelo) — pasos 19-20, columnas INS FINAL / INS F a
  considerar / Grupo del Archivo 10. Necesita el historial de REVA de cada tripulante, que no
  está confirmado como campo disponible en BigQuery. Sigue siendo 100% manual.
- **Los propios grupos de instructores** (AD2/AF2/AG2/AH2) — Fernando los escribe a mano cada mes;
  el notebook solo construye la fórmula que los lee, no decide su contenido.
- El destino "Rol Instructores... para asignar los vuelos a cada instructor" que menciona el
  documento, más allá de lo que ya hace este notebook (reemplazar el slot con el detalle de
  vuelos) — no se automatizó nada adicional ahí porque no se especificó otro formato/columna.
- Validación de capacidad por grupo (paso 21): la tabla auxiliar da la cantidad de vuelos por
  instructor; sumar por grupo y comparar contra un umbral sigue siendo una revisión manual (no
  hay un umbral fijo confirmado — el ejemplo del video, ~14/16/18, era específico de ese mes).
